# ⚠️ **Important Disclaimer**

Do **not edit or delete** any of the **Markdown cells** (the ones containing the questions and instructions).

Only write your answers in the **code cells provided below each question**.  
This ensures consistency during our feedback process.

### Q1. Load and Explore the Dataset

Load the `AirQalityDataset.csv` file into a pandas DataFrame using the correct separator.

After loading the data:

1. Display basic information about the dataset.
2. Save the statistical description of the dataset into a separate variable.
3. Drop fully empty/unnamed columns, and rows
4. Use `type()` to print the type of that description variable.

In [ ]:
import pandas as pd
import numpy as np
np.random.seed(0)

df = pd.read_csv("AirQualityDataset.csv", sep=';')
df = df.drop(columns=['Unnamed: 15', 'Unnamed: 16'], errors='ignore')
df = df.dropna(how='all')

info = df.info()
description = df.describe()
description_type = type(description)

<class 'pandas.core.frame.DataFrame'>
Index: 9357 entries, 0 to 9356
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           9357 non-null   object 
 1   Time           9357 non-null   object 
 2   CO(GT)         9357 non-null   float64
 3   PT08.S1(CO)    9357 non-null   float64
 4   NMHC(GT)       9357 non-null   float64
 5   C6H6(GT)       9357 non-null   float64
 6   PT08.S2(NMHC)  9357 non-null   float64
 7   NOx(GT)        9357 non-null   float64
 8   PT08.S3(NOx)   9357 non-null   float64
 9   NO2(GT)        9357 non-null   float64
 10  PT08.S4(NO2)   9357 non-null   float64
 11  PT08.S5(O3)    9357 non-null   float64
 12  T              9357 non-null   float64
 13  RH             9357 non-null   float64
 14  AH             9357 non-null   float64
dtypes: float64(13), object(2)
memory usage: 1.1+ MB


### Q2. Dataset structure and features overview  
Write a code to collect:
1. The number of rows and columns in the dataset.
2. The list of first 10 feature columns excluding `'Date'` and `'Time'`.  

Store both lists in tuple called `dataset_info` and print it.

In [ ]:
rows_cols = df.shape
features = df.drop(columns=['Date', 'Time'], errors='ignore').columns[:10].tolist()
dataset_info = (rows_cols, features)
dataset_info


((9357, 15),
 ['CO(GT)',
  'PT08.S1(CO)',
  'NMHC(GT)',
  'C6H6(GT)',
  'PT08.S2(NMHC)',
  'NOx(GT)',
  'PT08.S3(NOx)',
  'NO2(GT)',
  'PT08.S4(NO2)',
  'PT08.S5(O3)'])

### Q3. CO(GT) summary with Pandas and NumPy
Compute the **mean** and **standard deviation** of `CO(GT)` using both:
- Pandas
- NumPy

In [ ]:
valid_co = df[df['CO(GT)'] >= 0]['CO(GT)']
mean_pandas = valid_co.mean()
std_pandas = valid_co.std()
mean_numpy = np.mean(valid_co)
std_numpy = np.std(valid_co, ddof=1)

mean_pandas, std_pandas, mean_numpy, std_numpy


(np.float64(2.1527495439145166),
 1.4532520363373336,
 np.float64(2.1527495439145166),
 1.4532520363373336)

### Q4. Absolute humidity (AH) distribution  
Compute the **min**, **median**, and **max** of `AH` using Pandas.  

Do you notice an issue in the values?  
If you think that there are values that are problematic, replace them with the median of the column and print the same three statistics after that.


In [ ]:
first_value = df['AH'].agg(['min', 'median', 'max'])

median_AH = df['AH'].median()
df.loc[df['AH'] < 0, 'AH'] = median_AH
new_value = df['AH'].agg(['min', 'median', 'max'])

first_value, new_value

(min      -200.0000
 median      0.9768
 max         2.2310
 Name: AH, dtype: float64,
 min       0.1847
 median    0.9768
 max       2.2310
 Name: AH, dtype: float64)

### Q5. Humidity bands
Create a new column `humidity_band` using `RH`:
- `'dry'` if `RH < 30`
- `'comfortable'` if `30 <= RH <= 60`
- `'humid'` if `RH > 60`

Then show the **count** of each category.

In [ ]:
df['humidity_band'] = pd.cut(
    df['RH'],
    bins=[-float('inf'), 30, 60, float('inf')],
    labels=['dry', 'comfortable', 'humid']
)

humidity_counts = df['humidity_band'].value_counts()

humidity_counts

,count
humidity_band,
comfortable,4917
humid,2633
dry,1807


### Q6. Compute the Average 'CO(GT)' for Humid Conditions  

Using the `'humidity_band'` column created above, filter the dataset for rows labeled `'humid'` and compute the **average value of `'CO(GT)'`** for these observations.  

Format the output to 4 decimal places for better readability and precision.

In [ ]:
humid_condi = df.loc[df['humidity_band'] == 'humid', 'CO(GT)']
avg_co_humid = humid_condi.mean()
avg_co_humid_4dp = round(avg_co_humid, 4)
avg_co_humid_4dp

np.float64(-35.5567)

### Q7. Retrieve and sort array by a specific column
Create a NumPy array from the columns `[T, RH, AH]` (in this order), then sort the array by the **third column (`AH`)** ascending. Show the first 5 rows.

In [ ]:
arr = df[['T', 'RH', 'AH']].to_numpy()
sorted_arr = arr[arr[:, 2].argsort()]
sorted_arr[:5]


array([[ 0.    , 29.7   ,  0.1847],
       [11.8   , 13.5   ,  0.1862],
       [ 0.2   , 30.2   ,  0.191 ],
       [-0.1   , 31.9   ,  0.1975],
       [12.2   , 14.    ,  0.1988]])

### Q8. Normalized moisture index

Using the NumPy array you built above (**Do not change it**):  

1. Using Numpy, **Convert RH to a fraction** (0–1 scale) by dividing it by 100 and save it to another array `RH_frac`.
2. Using Numpy, **Compute a normalized moisture index** by dividing `AH` by `RH_frac`. This almost computes the amount of absolute humidity per unit of relative humidity.

Print the first 10 values of this new array and then **store** the result in the original DataFrame as a new column `'moisture_index'`.

In [ ]:
RH_frac = tri_array[:, 1] / 100
moisture_index = tri_array[:, 2] / RH_frac
moisture_index[:10]


array([1.54969325, 1.52096436, 1.38925926, 1.31116667, 1.32348993,
       1.32567568, 1.33855634, 1.28366667, 1.28107203, 1.2486711 ])

### Q9. Temperature profile for high moisture index  

Using Numpy only, and the `moisture_index` values you computed earlier:  

1. Find the **median** of `moisture_index`.  
2. Filter `tri_array` to include only rows where `moisture_index` is above this median.  
3. Compute and print the **mean temperature** for this high-moisture group using only NumPy.

Format the output to 4 decimal places for better readability and precision.

In [ ]:
median_moisture = np.median(moisture_index)
mask = moisture_index > median_moisture
mean_temprature_high = tri_array[mask, 0].mean()
mean_temprature_high = round(mean_temprature_high, 2)
mean_temprature_high


np.float64(25.23)

### Q10. Percentile-based filtering
Compute:
- the **85th percentile** of `C6H6(GT)` (benzene), and
- the **25th percentile** of `RH`.

Filter and return rows where `C6H6(GT)` is **above** its 85th percentile **and** `RH` is **below** its 25th percentile. Show the number of rows and the first 5 matches.


In [ ]:
percentile85th = np.percentile(df['C6H6(GT)'], 85)
percentile25th = np.percentile(df['RH'], 25)
filtered = df[(df['C6H6(GT)'] > percentile85th) & (df['RH'] < percentile25th)]
len(filtered), filtered.head()


(276,
            Date      Time  CO(GT)  PT08.S1(CO)  NMHC(GT)  C6H6(GT)  \
 137  16/03/2004  11.00.00     4.1       1571.0     327.0      20.0   
 138  16/03/2004  12.00.00     3.3       1452.0     283.0      18.3   
 139  16/03/2004  13.00.00     4.0       1579.0     366.0      22.3   
 140  16/03/2004  14.00.00     3.8       1466.0     318.0      20.4   
 144  16/03/2004  18.00.00     3.4       1447.0     237.0      17.8   
 
      PT08.S2(NMHC)  NOx(GT)  PT08.S3(NOx)  NO2(GT)  PT08.S4(NO2)  PT08.S5(O3)  \
 137         1297.0    314.0         730.0    162.0        1973.0       1729.0   
 138         1250.0    217.0         776.0    154.0        1868.0       1583.0   
 139         1359.0    252.0         724.0    161.0        1998.0       1671.0   
 140         1309.0    263.0         773.0    161.0        1897.0       1491.0   
 144         1235.0    184.0         859.0    139.0        1778.0       1296.0   
 
         T    RH      AH humidity_band  moisture_index  
 137  21.4  33.

### Q11. Simulate Sensor Measurement Noise and Analyze the Effect  

Simulate **normally distributed measurement noise** with a mean of `0` and a standard deviation of `100` (in raw sensor units). Then:  

- Use **NumPy** to generate the noise.  
- Use **Pandas** to add this noise to the `'PT08.S1(CO)'` column and store the result in a new column `'PT08.S1_noisy'`.  
- Print the **mean** and **standard deviation** of both `'PT08.S1(CO)'` and `'PT08.S1_noisy'` to observe the impact of the simulated noise.  

Observe how the added noise affects the distribution, particularly the spread (**standard deviation**). Format all printed values to **4 decimal places** using `.4f`.  


In [ ]:
noise = np.random.normal(loc=0, scale=100, size=len(df))

df['PT08.S1_noisy'] = df['PT08.S1(CO)'] + noise

mean_original = round(df['PT08.S1(CO)'].mean(), 4)
stdndard_original = round(df['PT08.S1(CO)'].std(), 4)

mean_noisy = round(df['PT08.S1_noisy'].mean(), 4)
stdndard_noisy = round(df['PT08.S1_noisy'].std(), 4)

(mean_original, stdndard_original, mean_noisy, stdndard_noisy)


(np.float64(1048.9901), 329.8327, np.float64(1049.8052), 343.9243)

# Make Your Results Reproducible

If you re-run the previous cell multiple times, you'll notice that the results involving randomness (e.g., simulated noise) change each time. This is because NumPy generates new random numbers on every execution.

To make your results **reproducible** (meaning that both you and your instructor get the **same output every time**) you need to set a fixed **random seed**.

As the final task, go back and add the following line to your code **immediately after importing NumPy** for the first time in your notebook:

```python
np.random.seed(0)


So, your NumPy import at the top of the notebook should now look like this:

```python
import numpy as np
np.random.seed(0)


# After Making This Change:

- Re-run **all cells** in the notebook from top to bottom.  
- Make sure **all outputs are visible**.  
- **Save your notebook.**  
- **Submit it as-is (with all outputs included.)**
